# Orchestrate Reg Prep Pipeline

Notebook wrapper for `RegPrepPipeline` (code-first config).

Pipeline steps in order:
1. `RequirementsExtractor` -> `*_requirements.json` + `*_requirements_extended.json`
2. `DeonticSlotExtractorLlama` -> stage1-3 report + stage4 slots (`*_requirements_extended_slots_llama3.json`)
3. `MainRequirementsSlotFilter` -> `*_requirements_slots_main.json` (must run after step 2)
4. `RequirementsGraphBuilder` -> graph JSON/GraphML from extended requirements
5. `graphVisualizer` -> HTML graph visualization

Note: if stage1-3 output is empty, stage4 fallback uses original per-node text to avoid empty slot output.


In [ ]:
from pathlib import Path
import json
import sys

project_root = Path('/Users/my/Documents/projects/detectionDeviation').expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from pipeline.Orchestrate_RegPrepPipeline import RegPrepPipeline

print('Project root:', project_root)


In [4]:
# -------------------------------
# Edit config here
# -------------------------------
reg_input_name = 'reg'   # folder under input/
reg_input_path = None    # optional explicit input path (.pdf or .json)
pdf_to_json = False      # True: run Docling PDF->JSON first

# main articles
# main use case 2 : 6, 12, 13, 15, 16, 17, 20, 21, 33, 34, 46, 49
# main use case 1: 6, 13, 14, 15, 17, 18, 20, 21, 46, 71
main_articles = [6, 13, 14, 15, 17, 18, 20, 21, 46, 71]
# 1-depth context additions
# for use case 2 : 8, 9, 10, 11, 14, 22, 23, 26, 30, 40, 42, 45, 47, 55, 63, 89, 92, 93
# for use case 1:: 8, 9, 10, 11, 12, 22, 23, 26, 40, 42, 45, 47, 49, 55, 63, 89, 93
context_articles = [8, 9, 10, 11, 12, 22, 23, 26, 40, 42, 45, 47, 49, 55, 63, 89, 93]

# keep True unless you want to skip a step
run_pdf_to_json = pdf_to_json
run_requirements_extractor = True
run_deontic_slot_extractor = True
run_main_slot_filter = True
run_graph_builder = True
run_graph_visualization = True


## Step-by-step run (recommended)

Use this when you want explicit visibility per step/output.


In [ ]:
pipeline = RegPrepPipeline(
    project_root=project_root,
    reg_input_name=reg_input_name,
    reg_input_path=reg_input_path,
    pdf_to_json=pdf_to_json,
    main_articles=main_articles,
    depth_one_articles=context_articles,
)

step_summary = {"project_root": str(project_root), "reg_input_name": reg_input_name, "steps": {}}

if run_pdf_to_json:
    step_summary["steps"]["docling_pdf_to_json"] = pipeline.run_docling_pdf_to_json()
else:
    step_summary["steps"]["docling_pdf_to_json"] = {"status": "skipped"}

if run_requirements_extractor:
    step_summary["steps"]["requirements_extractor"] = pipeline.run_requirements_extractor()
else:
    step_summary["steps"]["requirements_extractor"] = {"status": "skipped"}

if run_deontic_slot_extractor:
    step_summary["steps"]["deontic_slot_extractor"] = pipeline.run_deontic_slot_extractor()
else:
    step_summary["steps"]["deontic_slot_extractor"] = {"status": "skipped"}

# This is intentionally after deontic slot extraction
if run_main_slot_filter:
    step_summary["steps"]["main_requirements_slot_filter"] = pipeline.run_main_slot_filter()
else:
    step_summary["steps"]["main_requirements_slot_filter"] = {"status": "skipped"}

if run_graph_builder:
    step_summary["steps"]["graph_builder"] = pipeline.run_graph_builder()
else:
    step_summary["steps"]["graph_builder"] = {"status": "skipped"}

if run_graph_visualization:
    step_summary["steps"]["graph_visualization"] = pipeline.run_graph_visualization()
else:
    step_summary["steps"]["graph_visualization"] = {"status": "skipped"}

print(json.dumps(step_summary, indent=2, ensure_ascii=False))


## One-call run (optional)

Equivalent to the step-by-step run above, but executed through `pipeline.run(...)`.


In [5]:
pipeline = RegPrepPipeline(
    project_root=project_root,
    reg_input_name=reg_input_name,
    reg_input_path=reg_input_path,
    pdf_to_json=pdf_to_json,
    main_articles=main_articles,
    depth_one_articles=context_articles,
)

summary = pipeline.run(
    run_pdf_to_json=run_pdf_to_json,
    run_requirements_extractor=run_requirements_extractor,
    run_deontic_slot_extractor=run_deontic_slot_extractor,
    run_main_slot_filter=run_main_slot_filter,
    run_graph_builder=run_graph_builder,
    run_graph_visualization=run_graph_visualization,
)

print(json.dumps(summary, indent=2, ensure_ascii=False))


{
  "project_root": "/Users/my/Documents/projects/detectionDeviation",
  "reg_input_json": "/Users/my/Documents/projects/detectionDeviation/input/reg/gdpr.json",
  "steps": {
    "requirements_extractor": {
      "main_requirements": "/Users/my/Documents/projects/detectionDeviation/intermediate_results/reg/reg_requirements.json",
      "extended_requirements": "/Users/my/Documents/projects/detectionDeviation/intermediate_results/reg/reg_requirements_extended.json"
    },
    "deontic_slot_extractor": {
      "stage1_3_report": "/Users/my/Documents/projects/detectionDeviation/intermediate_results/reg/reg_requirements_extended_passive_active_report_llama3.json",
      "stage4_slots": "/Users/my/Documents/projects/detectionDeviation/intermediate_results/reg/reg_requirements_extended_slots_llama3.json"
    },
    "main_requirements_slot_filter": "/Users/my/Documents/projects/detectionDeviation/intermediate_results/reg/reg_requirements_slots_main.json",
    "graph_builder": {
      "nodes":